# Data Cleanup
### Clean up requirements
- Non consenting answers
- Students studying in Hungary
- Standardize nullish text answers
- Remove non valid answers
  - Negative answers in a "if yes" conditional question
  

In [ ]:
import re
import numpy as np
import pandas as pd
import logging

In [ ]:
# Import original csv
df = pd.read_csv("./datasets/forms-answers-dataset.csv")

In [ ]:

# --- Shape ---
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print()

# --- Column overview: name, dtype, non-null count, nulls, unique values ---
overview = df.describe()
display(overview)

# --- Duplicate rows ---
print(f"Duplicate rows: {df.duplicated().sum():,}")

In [ ]:
# --- First few rows ---
display(df.head())

### Rescue shifted answers before dropping unnamed columns
The first response (row 0) was submitted on an earlier version of the form: its 11 answers for the
Partner / Family / Friends support items were saved in the trailing `Unnamed` columns instead of their own columns.

In [ ]:
unnamed_cols = df.filter(regex=r'(?i)unnamed').columns
shifted_rows = df.index[df[unnamed_cols].notna().any(axis=1)]
print("Rows with values in unnamed columns:", shifted_rows.tolist())

# Columns 87-97 are the 11 support items that were left empty for those rows
support_cols = df.columns[87:98]
assert df.loc[shifted_rows, support_cols].isna().all().all()
df.loc[shifted_rows, support_cols] = df.loc[shifted_rows, unnamed_cols].astype(int).astype(str).values

### Clean unnamed columns

In [ ]:
# Removing unnamed columns
to_drop = df.filter(regex=r'(?i)unnamed').columns.tolist()
print("Dropping:", to_drop)
df = df.drop(columns=to_drop)

### Renamed columns to numerical values

In [ ]:
# Save original column names
original_columns = df.columns.tolist()

# Renames columns to be numerical
df.columns = range(df.shape[1])

In [ ]:
display(df.head(2))

### Remove invalid respondents
- Respondents who did not consent
- The researcher's test submission (asked not to be included)

In [ ]:
n_before = len(df)

non_consenting = df[1].ne("Yes")
print("Non consenting:", non_consenting.sum())
df = df[~non_consenting]

test_submission = df[2].str.contains("do not include my answers", case=False, na=False)
print("Test submissions:", test_submission.sum())
df = df[~test_submission]

df = df.reset_index(drop=True)
print(f"Rows: {n_before} -> {len(df)}")

### Basic text tidy-up
Trim whitespace, unify apostrophes / non-breaking spaces, turn empty strings into null.

In [ ]:
text_cols = df.select_dtypes(include=["object", "string"]).columns

for col in text_cols:
    df[col] = (
        df[col]
        .str.replace("\u00a0", " ", regex=False)
        .str.replace("’", "'", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", np.nan)
    )

### Standardize nullish text answers
Open-ended answers that carry no information (`.`, `-`, `/`, `?`, `asdf`, `none`, `idk`, `no`, ...) become null (`NaN`).
Null is used instead of the text "N/A" so pandas treats them as missing in counts, means and filters.

In [ ]:
OPEN_ENDED = [5, 6, 7, 12, 13, 14, 22, 54, 69, 100, 101]

NULLISH = {
    "n/a", "na", "null", "none", "none;", "nothing", "no", "nope", "non", "idk", "asdf", "aassf", "s<xc",
    "i don't know", "i dont know", "i don't now", "not sure", "yea", "now", "ninguna", "немає", "ethnicity",
    "i don't understand what this means...", "я не задавався цим питанням", "human", "людина",
}

def is_nullish(value):
    if pd.isna(value):
        return False
    v = str(value).strip().lower()
    return v in NULLISH or re.fullmatch(r"[\W_]*", v) is not None  # only punctuation / symbols

for col in OPEN_ENDED:
    mask = df[col].map(is_nullish)
    if mask.any():
        print(f"[{col}] -> null: {sorted(df.loc[mask, col].unique())}")
    df.loc[mask, col] = np.nan

### Standardize categorical answers
Older form versions and "Other" free text produced variants of the same option.

In [ ]:
# Age
df[3] = df[3].str.replace(r"\s*-\s*", "-", regex=True).str.replace(r"\s*\+", "+", regex=True)

# Gender: drop free-text add-ons, keep the selected option
df[4] = df[4].replace({
    "Man;theres only 2 genders": "Man",
    "Non-binary;I go by she/they, if I could say being a butch lesbian was my gender, then that would be my most accurate answer": "Non-binary",
})

# Identity: older wording of the same option
df[9] = df[9].str.replace("(you received a medical diagnosis)", "(you have received a medical diagnosis)", regex=False)

# Diagnosis timing: older form options
df[11] = df[11].replace({"Yes, during vet school": "During vet school", "No": "I have not received any"})

# Professional support: remove "None" when combined with other options
df[13] = (
    df[13].str.split(";")
    .map(lambda parts: ";".join(p.strip() for p in parts if p.strip() and p.strip().lower() != "none") if isinstance(parts, list) else parts)
    .replace("", np.nan)
)

# Challenge ranking: "Moderate" only existed in an older form version, equivalent to "Somewhat"
CHALLENGE_COLS = list(range(29, 41))
df[CHALLENGE_COLS] = df[CHALLENGE_COLS].replace("Moderate challenging", "Somewhat challenging")

# Psychiatrist (Yes/No question): a stray "5" from the older form
df.loc[~df[67].isin(["Yes", "No"]), 67] = np.nan

for col in [3, 4, 9, 11, 67]:
    print(col, df[col].value_counts(dropna=False).to_dict(), "\n")

### Country of study

In [ ]:
COUNTRY_PATTERNS = [
    ("Hungary", r"hung|hubgary|budapest"),
    ("Ukraine", r"ukr|україн|^uk$"),
    ("Romania", r"rom|roum"),
    ("France", r"fran|french|nantes"),
    ("Spain", r"spain|españa"),
    ("Poland", r"pol|wroc"),
    ("Portugal", r"portug"),
    ("Brazil", r"bra[sz]il"),
    ("Bulgaria", r"bulgar"),
    ("Finland", r"finl"),
    ("Italy", r"ital"),
]

def clean_country(value):
    if pd.isna(value):
        return np.nan
    for country, pattern in COUNTRY_PATTERNS:
        if re.search(pattern, value.lower()):
            return country
    logging.warning(f"Unmapped country: {value!r}")
    return value

df[7] = df[7].map(clean_country)
print(df[7].value_counts(dropna=False))

### Nationality
Single nationalities are mapped to one spelling; multiple nationalities are kept as written.

In [ ]:
NATIONALITY_MAP = {
    "French": ["french", "française", "francaise", "france"],
    "Spanish": ["spanish", "spain", "española", "español"],
    "Irish": ["irish", "ireland", "irisí"],
    "Finnish": ["finnish", "finland", "fin", "finnish citizenship"],
    "Portuguese": ["portuguese"],
    "Ukrainian": ["ukrainian", "ukranian", "ukraine", "uk", "україна", "українка", "українець", "українське"],
    "German": ["german", "germany"],
    "Romanian": ["romanian"],
    "Norwegian": ["norwegian", "norway"],
    "Austrian": ["austrian", "austria"],
    "Hungarian": ["hungarian"],
    "South African": ["south african", "south africa"],
    "Bulgarian": ["bulgarian", "bulgaria"],
    "Brazilian": ["brazilian", "brasilian", "brasil"],
    "American": ["usa"],
    "Swiss": ["swiss", "switzerland"],
    "Indian": ["indian", "india"],
    "Iraqi": ["iraq"],
    "Moroccan": ["moroccan", "morrocan"],
    "Chinese": ["chinese"],
}
nationality_lookup = {alias: name for name, aliases in NATIONALITY_MAP.items() for alias in aliases}

df[5] = df[5].map(lambda v: nationality_lookup.get(v.lower(), v) if isinstance(v, str) else v)
print(df[5].value_counts(dropna=False).to_string())

### Ethnicity
Self-identified, so only spelling, casing and translations are unified (no merging of categories).

In [ ]:
ETHNICITY_MAP = {
    "caucasien": "Caucasian",
    "blanca": "White", "blanco": "White", "branca": "White",
    "українка": "Ukrainian", "ukranian": "Ukrainian", "ukraine": "Ukrainian", "ukrainian lengvich": "Ukrainian",
    "norewgian": "Norwegian",
    "france": "French",
    "austria": "Austrian",
    "bulgaria": "Bulgarian",
}

def clean_ethnicity(value):
    if pd.isna(value):
        return np.nan
    if value.lower() in ETHNICITY_MAP:
        return ETHNICITY_MAP[value.lower()]
    if value.islower() or value.isupper():
        return value.title()
    return value

df[6] = df[6].map(clean_ethnicity)
print(df[6].value_counts(dropna=False).head(15).to_string())

### "If yes" conditional questions
Answers to a follow-up question are only valid if the respondent answered the trigger question positively.
Any other answer (e.g. "No", "I said no before", "Never heard of it") becomes null.

In [ ]:
def null_if_not(target_col, valid_mask, label):
    invalid = ~valid_mask & df[target_col].notna()
    print(f"[{target_col}] {label}: {invalid.sum()} invalid answers -> null")
    if invalid.any():
        print("   ", df.loc[invalid, target_col].value_counts().to_dict())
    df.loc[invalid, target_col] = np.nan

# 22 "If yes, what was said?" <- 21 professors discussed neurodivergence (Yes / Shortly)
null_if_not(22, df[21].isin(["Yes", "Shortly"]), "what was said")

# 12 "If you answered 'I have not received any', why no diagnosis?" <- 11
null_if_not(12, df[11].eq("I have not received any"), "why no diagnosis")

# 54 "If 'Yes' in any of the previous couple of questions: what strengths?" <- 52 or 53
null_if_not(54, df[52].eq("Yes") | df[53].eq("Yes"), "what strengths")

# 69 "If yes, was NOMV formally discussed in your University?" <- 68 heard of NOMV
null_if_not(69, df[68].eq("Yes"), "NOMV discussed")

In [ ]:
# Remaining "Other" answers for NOMV from respondents who did hear of it, mapped to the closest option
df[69] = df[69].replace({
    "Associations made by students": "Not by the university but by the students",
    "Yes, in a Lecture held by a guest speaker": "Yes but briefly",
    "Career eve": "Yes but briefly",
    "Heard it from the norwegian vet who studied mental health/ suicide rates amongst veterinarians, on FB": "Not at all",
    "In french vet school, a lot of suicides happens this year and it was related by medias. That is how I heard about it.": "Not at all",
})
print(df[69].value_counts(dropna=False))

### Neurodivergent-only questions
These questions say *"If you don't consider yourself neurodivergent, skip this question"*.
Answers from respondents who identified **only** as Neurotypical become null.

In [ ]:
ND_ONLY_COLS = [18, 19, 28, *CHALLENGE_COLS, 52, 53, 54]

neurotypical_only = df[9].eq("Neurotypical")
answered = df.loc[neurotypical_only, ND_ONLY_COLS].notna().any(axis=1)
print(f"Neurotypical-only respondents who answered ND-only questions: {answered.sum()}")
df.loc[neurotypical_only, ND_ONLY_COLS] = np.nan

### Convert scales to numbers
- Over the last 2 weeks (PHQ-9 / GAD-7 items, cols 70-85): `"1 - Several days"` / `"1 - Some days"` -> `1`
- Partner / Family / Friends support (MSPSS items, cols 86-97): `"7 Very strongly agree"` -> `7`
- Float rating columns that only hold whole numbers -> nullable integers

In [ ]:
PHQ_GAD_COLS = list(range(70, 86))
MSPSS_COLS = list(range(86, 98))

for col in PHQ_GAD_COLS + MSPSS_COLS:
    df[col] = pd.to_numeric(df[col].str.extract(r"^(\d)", expand=False)).astype("Int64")

for col in df.select_dtypes(include="float").columns:
    df[col] = df[col].astype("Int64")

# Sanity check ranges
RANGES = {(0, 3): PHQ_GAD_COLS, (1, 7): MSPSS_COLS,
          (1, 5): [16, 17, *range(23, 29), *range(41, 52), *range(55, 65), 98, 99]}
for (lo, hi), cols in RANGES.items():
    values = df[cols].stack().dropna()
    assert values.between(lo, hi).all(), f"Out of range values in {cols}"
print("All numeric scales within range")

### Timestamp

In [ ]:
TZ_OFFSETS = {"EET": "+02:00", "EEST": "+03:00"}
ts = df[0].str.replace(r"\b(EEST|EET)$", lambda m: TZ_OFFSETS[m.group(1)], regex=True)
df[0] = pd.to_datetime(ts, format="%Y/%m/%d %I:%M:%S %p %z", utc=True)
print(df[0].min(), "->", df[0].max())

### Flags for review
Not changed, only reported.

In [ ]:
# Contradictory identity selections (neurotypical together with neurodivergent)
contradictory = df[9].str.contains("Neurotypical", na=False) & df[9].str.contains("neurodivergent", na=False)
print(f"Neurotypical + neurodivergent selected together: {contradictory.sum()}")

# Same non-anonymous name submitted more than once (possible re-submissions)
names = df[2].str.lower()
named = names[~names.str.match(r"^(anonym|anónim|анонім|null)", na=True)]
print("Repeated names:", sorted(named[named.duplicated(keep=False)].unique()))

### Export
Respondent names are dropped from the exported dataset (not needed for analysis, personal data).
A codebook maps the numeric column ids back to the original questions.

In [ ]:
clean = df.drop(columns=[2])
clean.to_csv("./datasets/forms-answers-cleaned.csv", index=False)

codebook = pd.DataFrame({"id": range(len(original_columns)), "question": original_columns})
codebook.to_csv("./datasets/codebook.csv", index=False)

print(f"Saved {clean.shape[0]} rows x {clean.shape[1]} columns")
display(clean.head())

In [ ]:
# Subset: only students who study / studied in Hungary
hungary = clean[clean[7].eq("Hungary")]
hungary.to_csv("./datasets/forms-answers-cleaned-hungary.csv", index=False)
print(f"Saved {hungary.shape[0]} rows x {hungary.shape[1]} columns (Hungary only)")